<a href="https://colab.research.google.com/github/xperex/Meshroom_-_GPU_for_Photogrammetry.ipynb/blob/main/Meshroom_%2B_GPU_for_Photogrammetry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

(from https://gist.github.com/donmahallem/22134574382b7bd8a67c1550734fcfc4 . Thanks to Ryan Baumann for sharing the gist on Mastodon.)

Make sure you have GPU runtime. If you run the code below and receive the error, 'GPU device not found', click on 'Runtime' in the menu at top, 'change runtime type', >> 'select hardware acceleration' and select GPU. Then re-run the code to make sure it's running.

In [1]:
import tensorflow as tf
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
  raise SystemError('GPU device not found')
print('Found GPU at: {}'.format(device_name))

Found GPU at: /device:GPU:0


Get Meshroom and some data.

In [ ]:
# IF YOU HAVE A FOLDER WITH IMAGES IN YOUR GOOGLE DRIVE
# you can connect this notebook to your drive by using
# this code:
from google.colab import drive
drive.mount('/content/drive')

# and then your files will be available. You can click the
# > at top left and select 'files' to see the file tree.
# right-clicking on a file or folder will allow you to copy the full path
# but nb if you then paste that path into any code, you need to remove the
# `/content/` part of the path.

In [2]:
# or you can run this cell to upload your files.

from google.colab import files

# optional upload for the meshfile

uploaded = files.upload()

for fn in uploaded.keys():
    print('User uploaded file "{name}" with length {length} bytes'.format( name=fn, length=len(uploaded[fn])))

Saving djidji_fly_20260605_171232_0014_1780701822160_video_000001.jpg to djidji_fly_20260605_171232_0014_1780701822160_video_000001.jpg
Saving djidji_fly_20260605_171232_0014_1780701822160_video_000002.jpg to djidji_fly_20260605_171232_0014_1780701822160_video_000002.jpg
Saving djidji_fly_20260605_171232_0014_1780701822160_video_000003.jpg to djidji_fly_20260605_171232_0014_1780701822160_video_000003.jpg
Saving djidji_fly_20260605_171232_0014_1780701822160_video_000004.jpg to djidji_fly_20260605_171232_0014_1780701822160_video_000004.jpg
Saving djidji_fly_20260605_171232_0014_1780701822160_video_000005.jpg to djidji_fly_20260605_171232_0014_1780701822160_video_000005.jpg
Saving djidji_fly_20260605_171232_0014_1780701822160_video_000006.jpg to djidji_fly_20260605_171232_0014_1780701822160_video_000006.jpg
Saving djidji_fly_20260605_171232_0014_1780701822160_video_000007.jpg to djidji_fly_20260605_171232_0014_1780701822160_video_000007.jpg
Saving djidji_fly_20260605_171232_0014_178070182

If you upload files, make sure to move them into their own folder. Either upload as a zip file and then use `!unzip folder.zip` or use eg. `mv *.jpg /my_dataset/`

In [33]:
mv *.jpg /my_dataset/

mv: target '/my_dataset/' is not a directory


In [35]:
!mkdir my_dataset
!mv *.jpg my_dataset/

mkdir: cannot create directory ‘my_dataset’: File exists


In [26]:
# or you can grab some data from AliceVision for the time being.
!git clone https://github.com/alicevision/dataset_buddha


Cloning into 'dataset_buddha'...
remote: Enumerating objects: 537, done.
remote: Total 537 (delta 0), reused 0 (delta 0), pack-reused 537 (from 1)
Receiving objects: 100% (537/537), 762.00 MiB | 19.20 MiB/s, done.
Resolving deltas: 100% (147/147), done.
Updating files: 100% (223/223), done.


In [27]:
# get Meshroom
!wget -N https://github.com/alicevision/meshroom/releases/download/v2019.1.0/Meshroom-2019.1.0-linux.tar.gz
!mkdir meshroom
!tar xzf Meshroom-2019.1.0-linux.tar.gz -C ./meshroom

--2026-06-09 01:18:19--  https://github.com/alicevision/meshroom/releases/download/v2019.1.0/Meshroom-2019.1.0-linux.tar.gz
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/34405381/ac2a6000-44ad-11e9-9c7e-7405269e659a?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-06-09T02%3A03%3A30Z&rscd=attachment%3B+filename%3DMeshroom-2019.1.0-linux.tar.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-06-09T01%3A03%3A22Z&ske=2026-06-09T02%3A03%3A30Z&sks=b&skv=2018-11-09&sig=qUw1Mzvupuzl8qRUYcEBnbPr8K1Cj7SMS077jho7OSY%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4MDk3MTM1NiwibmJmIjoxNzgwOTY3NzU2LCJwYXRoI

# Meshing!

Make sure that the next line points to the correct folder, eg my_dataset if you uploaded your own, or the dataset_buddha if you used wget to get the Buddha pictures from Alicevision. (and the line below would be chanted eg `--input ./dataset_buddha/buddha --output`)

In [ ]:
!mkdir ./object_out
!./meshroom/Meshroom-2019.1.0/meshroom_photogrammetry --input ./my_dataset/ --output ./object_out

mkdir: cannot create directory ‘./object_out’: File exists
Plugins loaded:  CameraCalibration, CameraInit, CameraLocalization, CameraRigCalibration, CameraRigLocalization, ConvertSfMFormat, DepthMap, DepthMapFilter, ExportAnimatedCamera, ExportMaya, FeatureExtraction, FeatureMatching, ImageMatching, ImageMatchingMultiSfM, KeyframeSelection, MeshDecimate, MeshDenoising, MeshFiltering, MeshResampling, Meshing, PrepareDenseScene, Publish, SfMAlignment, SfMTransform, StructureFromMotion, Texturing
Program called with the following parameters:
 * allowSingleView = 1
 * defaultCameraModel = "" (default)
 * defaultFieldOfView = 45
 * defaultFocalLengthPix = -1 (default)
 * defaultIntrinsic = "" (default)
 * groupCameraFallback =  Unknown Type "20EGroupCameraFallback"
 * imageFolder = "" (default)
 * input = "/tmp/tmpw8xo9txf/CameraInit/c448939571d5c70b05c9ae4ad416ada37d6273ca//viewpoints.sfm"
 * output = "/tmp/tmpw8xo9txf/CameraInit/c448939571d5c70b05c9ae4ad416ada37d6273ca/cameraInit.sfm"
 * 

In [25]:
# zip and download the results!
!zip -r meshobject.zip ./object_out
files.download('meshobject.zip')

updating: object_out/ (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>